# Folder to CBZ Converter

Scans a directory for subfolders. For each subfolder that contains images, this notebook:

1. Creates a `.cbz` (a zip file) named after the folder, containing the images.
2. Deletes the original folder after the archive is created successfully.

Set `ROOT_DIR` below, and keep `DRY_RUN = True` first to preview what will happen before anything is deleted.

In [1]:
import shutil
import zipfile
from pathlib import Path

# --- Configuration ---
ROOT_DIR = Path(r"Z:\Misc")  # directory containing the subfolders to scan
DRY_RUN = True  # set to False to actually create CBZ files and delete folders
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".webp", ".tiff", ".avif"}

In [2]:
def find_images(folder: Path):
    """Return sorted list of image files directly inside folder (non-recursive)."""
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def has_subdirectories(folder: Path) -> bool:
    return any(p.is_dir() for p in folder.iterdir())


def create_cbz(folder: Path, images: list[Path]) -> Path:
    """Create a .cbz archive named after folder, containing the given images."""
    cbz_path = folder.parent / f"{folder.name}.cbz"
    with zipfile.ZipFile(cbz_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for image in images:
            zf.write(image, arcname=image.name)
    return cbz_path


def verify_cbz(cbz_path: Path, expected_count: int) -> bool:
    """Confirm the archive is readable, uncorrupted, and has all expected entries."""
    with zipfile.ZipFile(cbz_path, "r") as zf:
        if zf.testzip() is not None:
            return False
        return len(zf.namelist()) == expected_count


def process_directory(root_dir: Path, dry_run: bool = True, skip_if_nested_dirs: bool = True):
    """Scan root_dir for subfolders with images, archive them as CBZ, then delete the folder."""
    if not root_dir.is_dir():
        raise NotADirectoryError(f"ROOT_DIR does not exist or is not a directory: {root_dir}")

    for folder in sorted(p for p in root_dir.iterdir() if p.is_dir()):
        images = find_images(folder)
        if not images:
            print(f"Skipping (no images): {folder}")
            continue

        if has_subdirectories(folder):
            message = f"{folder} contains subfolders whose contents would NOT be archived"
            if skip_if_nested_dirs:
                print(f"Skipping ({message}): {folder}")
                continue
            print(f"Warning: {message}")

        cbz_path = folder.parent / f"{folder.name}.cbz"
        if cbz_path.exists():
            print(f"Skipping (target archive already exists): {cbz_path}")
            continue

        if dry_run:
            print(f"[DRY RUN] Would create {cbz_path} from {len(images)} image(s), then delete {folder}")
            continue

        try:
            created_path = create_cbz(folder, images)
            if not verify_cbz(created_path, len(images)):
                raise RuntimeError(f"Verification failed for {created_path}; original folder was NOT deleted")
        except Exception as exc:
            print(f"Error processing {folder}: {exc}")
            created_path.unlink(missing_ok=True)
            continue

        print(f"Created {created_path} from {len(images)} image(s)")
        shutil.rmtree(folder)
        print(f"Deleted folder {folder}")

In [3]:
process_directory(ROOT_DIR, dry_run=False)

Skipping (no images): Z:\Misc\2JIMUSUBI (Ohno Kanae)
Skipping (no images): Z:\Misc\3104tyome (3104)
Skipping (no images): Z:\Misc\54BURGER (Marugoshi)
Skipping (no images): Z:\Misc\70 Nenshiki Yuukyuu Kikan (Ohagi-san)
Created Z:\Misc\[Amakuchi Syoujo (Umakuchi Syouyu)] Shigure Make Love (Kantai Collection -KanColle-) [English] [Digital].cbz from 29 image(s)
Deleted folder Z:\Misc\[Amakuchi Syoujo (Umakuchi Syouyu)] Shigure Make Love (Kantai Collection -KanColle-) [English] [Digital]
Created Z:\Misc\[Denki Neko (Toku)] Dekaketsu PowerHara Tenchou ni Shiboraretai! 2 [English] [Digital].cbz from 89 image(s)
Deleted folder Z:\Misc\[Denki Neko (Toku)] Dekaketsu PowerHara Tenchou ni Shiboraretai! 2 [English] [Digital]
Skipping (no images): Z:\Misc\ABBB
Skipping (no images): Z:\Misc\Airandou
Skipping (no images): Z:\Misc\Aizome Gorou
Skipping (no images): Z:\Misc\Aka Seiryuu
Skipping (no images): Z:\Misc\Akashiba Honpo (Igusa)
Skipping (no images): Z:\Misc\Akino Sora
Skipping (no images): Z: